# 图像拼接项目

图像拼接的步骤：
- 第一步，读取文件，将图片设置成一样大小640x480
- 第二步，找特征点，描述子，计算单应性矩阵
- 第三步，根据单应性矩阵对图像进行变换，然后平移
- 第四步，拼接并输出最终结果

## 0.导入库

In [ ]:
import cv2 as cv              # OpenCV计算机视觉库
import numpy as np            # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.show()

## 1.读取图片，并统一图片大小

In [ ]:
img1 = cv.imread('map1.png')  # 读取第一张地图图像
img2 = cv.imread('map2.png')  # 读取第二张地图图像

# resize统一尺寸为640x480，参数顺序为(width, height)
img1 = cv.resize(img1, (640, 480))
img2 = cv.resize(img2, (640, 480))

inputs = np.hstack((img1, img2))  # 水平拼接两张图，便于对比查看
show(inputs)

## 2.找特征点，描述子，计算单应性矩阵

- 1.创建特征转换对象(SIFT、ORB等)
- 2.通过特征转换对象获得特征点和描述符
- 3.创建特征匹配器
- 4.进行特征匹配
- 5.过滤特征，找出有效的特征匹配点
- 6.计算单应性矩阵

In [ ]:
def get_homo(img1, img2):
    """计算两张图像之间的单应性矩阵(Homography Matrix)"""
    
    # 1.创建ORB特征检测器（比SIFT更快，适合实时应用）
    orb = cv.ORB_create()

    # 2.检测特征点并计算描述符
    k1, d1 = orb.detectAndCompute(img1, None)  # k1=关键点列表，d1=描述符数组
    k2, d2 = orb.detectAndCompute(img2, None)

    # 3.创建暴力匹配器（NORM_HAMMING用于ORB的二进制描述符）
    bf = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=False)

    # 4.knnMatch：对每个描述符找k=2个最近邻匹配，用于Lowe's ratio test
    matches = bf.knnMatch(d1, d2, k=2)

    # 5.Lowe's ratio test：过滤掉不可靠的匹配
    verify_ratio = 0.8   # 如果最佳匹配距离 < 0.8 * 次佳匹配距离，则认为是好匹配
    verify_matches = []
    for m1, m2 in matches:
        if m1.distance < verify_ratio * m2.distance:
            verify_matches.append(m1)

    # 6.用RANSAC算法计算单应性矩阵（至少需要4个匹配点对）
    min_matches = 8
    if len(verify_matches) > min_matches:
        img1_pts = []
        img2_pts = []

        for m in verify_matches:
            img1_pts.append(k1[m.queryIdx].pt)  # queryIdx对应img1的特征点索引
            img2_pts.append(k2[m.trainIdx].pt)  # trainIdx对应img2的特征点索引

        # reshape为findHomography所需的形状：(N, 1, 2)
        img1_pts = np.float32(img1_pts).reshape(-1, 1, 2)
        img2_pts = np.float32(img2_pts).reshape(-1, 1, 2)

        # findHomography：RANSAC方法计算3x3单应性矩阵，reprojThreshold=5.0
        H, mask = cv.findHomography(img1_pts, img2_pts, cv.RANSAC, 5.0)

        return H
    else:
        print('err: Not enough matches!')
        exit()

In [ ]:
H = get_homo(img1, img2)  # 获取img1到img2的单应性矩阵（3x3变换矩阵）

## 3.根据单应性矩阵对图像进行变换，然后平移

- 1.获得原始图的高/宽
- 2.获得每张图片的四个角点
- 3.对图片进行变换（单应性矩阵使图进行旋转，平移）
- 4.创建一张大图，将两张图拼接到一起
- 5.将结果输出

In [ ]:
def stitch_image(img1, img2, H):
    """根据单应性矩阵H将img1变换到img2的坐标系，然后拼接两张图"""
    
    # 1.获取两张原始图的高/宽
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]

    # 2.定义两张图片的四个角点坐标（左上、左下、右下、右上），reshape为(N,1,2)
    img1_dims = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
    img2_dims = np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)

    # 3.1 透视变换：用单应性矩阵H将img1的角点变换到img2的坐标系
    img1_transform = cv.perspectiveTransform(img1_dims, H)

    # 3.2 计算拼接后大图的边界范围
    result_dims = np.concatenate((img2_dims, img1_transform), axis=0)  # 合并两图所有角点

    [x_min, y_min] = np.int32(result_dims.min(axis=0).ravel() - 0.5)  # 最小坐标
    [x_max, y_max] = np.int32(result_dims.max(axis=0).ravel() + 0.5)  # 最大坐标

    # 平移距离：将负坐标移到正区间，确保所有像素都在画布内
    transform_dist = [-x_min, -y_min]

    # 构造平移矩阵（3x3），将图像整体平移到正坐标区域
    transform_array = np.array([[1, 0, transform_dist[0]],
                                [0, 1, transform_dist[1]],
                                [0, 0, 1]])

    # 对img1做透视变换+平移，生成与拼接结果同尺寸的画布
    result_img = cv.warpPerspective(
        img1,
        transform_array.dot(H),  # 矩阵乘法：平移矩阵 x 单应性矩阵
        (x_max - x_min, y_max - y_min)  # 输出画布尺寸
    )

    # 4.将img2放到画布的对应位置（平移后的区域）
    result_img[transform_dist[1]:transform_dist[1] + h2,
               transform_dist[0]:transform_dist[0] + w2] = img2

    return result_img

In [ ]:
result_image = stitch_image(img1, img2, H)  # 执行图像拼接
show(result_image)